# Question → JSON plan translator: training on Colab

Fine-tunes a small model (LoRA) to turn a short question into a JSON plan, using `train.py` and
generated training data.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. Checkpoints go to Google Drive every 200 steps. If the session drops, just **Run all** again:
   training continues from the last checkpoint, it does not start over.
3. At the end, `predictions.jsonl` downloads. The app scores it.

In [ ]:
# Settings
MODEL = "Qwen/Qwen3-0.6B"      # google/gemma-3-270m-it needs a Hugging Face login and its licence accepted
RUN_NAME = "qwen06-full-v1"    # change this to start a fresh run instead of continuing this one
BATCH = 8

import os
from google.colab import drive
drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/plan-translator/{RUN_NAME}"
os.makedirs(os.path.dirname(OUT), exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The library versions train.py was tested with
!pip install -q transformers==5.22.0 trl==1.17.0 peft==0.21.2 datasets==5.1.0 accelerate==1.15.0

In [ ]:
# Scripts and data
!rm -rf /content/repo && git clone -q --depth 1 https://github.com/fatihsokmen/plan-translator-colab /content/repo
!tar xzf /content/repo/data.tar.gz -C /content/repo && wc -l /content/repo/data/*/*.jsonl

In [ ]:
# Speed check: 30 short steps, then an estimate for the full run (this Mac: about 7 s per step)
import math, re
!cd /content/repo && python train.py --model {MODEL} --data data --out /content/speedtest --limit 400 --max_steps 30 --bs {BATCH} --dev_rows 5 --eval_rows 20 > /content/speedtest.log 2>&1
log = open("/content/speedtest.log").read()
m = re.search(r"'train_runtime': '?([\d.]+)", log)
if m:
    per_step = float(m.group(1)) / 30
    steps = math.ceil(sum(1 for _ in open("/content/repo/data/train/train.jsonl")) / BATCH)
    print(f"{per_step:.2f} s per step → full run about {steps * per_step / 3600:.1f} h ({steps} steps)")
else:
    print(log[-3000:])

In [ ]:
# Full training (all training rows, 1 epoch). Prints the dev exact-plan rate at the end.
!cd /content/repo && python train.py --model {MODEL} --data data --out {OUT} --bs {BATCH} --save_steps 200 2>&1 | tee -a {OUT}.log

In [ ]:
# Plans for the frozen test prompts; the app scores this file
!cd /content/repo && python predict.py --model {OUT}/final --prompts data/test/real_like_v2.prompts.jsonl --out {OUT}/predictions.jsonl
from google.colab import files
files.download(f"{OUT}/predictions.jsonl")